In [ ]:
import sys
sys.path.append(r'../src')
from preprocessing import load_raster
import numpy as np

data, transform, pixel_area = load_raster(r'../data/landuse_2023.tif')
print("数据形状:", data.shape)
print("像元面积（平方米）:", pixel_area)

In [ ]:
import sys
sys.path.append(r'../src')
from preprocessing import load_raster,classify_ph
import numpy as np
ph_data, _, _ = load_raster(r'../data/pH_0-5cm_1km.tif')
ph_data = ph_data/100
ph_class = classify_ph(ph_data)

print("pH数据形状:", ph_data.shape)
print("pH值范围: {:.2f} - {:.2f}".format(np.nanmin(ph_data), np.nanmax(ph_data)))
for i in range(1, 6):
    print(f"  等级 {i}: {np.sum(ph_class == i)} 个像元")

In [ ]:
import sys
sys.path.append(r'../src')
from preprocessing import load_raster,reclassify_landuse

lu_path = r'../data/landuse_2023.tif'
lu_data, _, _ = load_raster(lu_path)

landuse_names,labels, unique_classes = reclassify_landuse(lu_data)
print("土地利用类别:", labels)

In [ ]:
import sys
sys.path.append(r'../src')
from preprocessing import load_raster, classify_ph, reclassify_landuse, calc_area_crosstab
import numpy as np

# 1. 读取土地利用数据
lu_path = r'../data/landuse_2023.tif'
lu_data, lu_transform, lu_pixel_area = load_raster(lu_path)
print("土地利用数据形状:", lu_data.shape)
print("土地利用像元面积（m²）:", lu_pixel_area)

# 2. 读取pH数据
ph_path = r'../data/pH_0-5cm_1km.tif'
ph_data, ph_transform, ph_pixel_area = load_raster(ph_path)
ph_data = ph_data/100
print("pH数据形状:", ph_data.shape)
print("pH像元面积（m²）:", ph_pixel_area)

# 3. pH分级
ph_class = classify_ph(ph_data)
print("\npH分级统计：")
for i in range(1, 6):
    print(f"  等级{i}: {np.sum(ph_class == i)} 像元")

# 4. 土地利用重分类
landuse_names, labels, unique_classes = reclassify_landuse(lu_data)
print("\n土地利用类别:", labels)

# 5. 检查两个栅格是否对齐
print("\n栅格对齐检查：")
print(f"  土地利用形状: {lu_data.shape}, pH形状: {ph_data.shape}")
if lu_data.shape == ph_data.shape:
    print("  ✓ 形状一致，可以直接做交叉统计")
else:
    print("  ✗ 形状不一致，需要重采样后再做交叉统计")

In [ ]:
import rasterio

lu_path = r'../data/landuse_2023.tif'
ph_path = r'../data/pH_0-5cm_1km.tif'

with rasterio.open(lu_path) as lu, rasterio.open(ph_path) as ph:
    print("土地利用投影:", lu.crs)
    print("土地利用范围:", lu.bounds)
    print("土地利用分辨率:", lu.res)
    print("土地利用形状:", lu.shape)
    print()
    print("pH投影:", ph.crs)
    print("pH范围:", ph.bounds)
    print("pH分辨率:", ph.res)
    print("pH形状:", ph.shape)

In [ ]:
import sys
sys.path.append(r'../src')

from preprocessing import resample_raster

ph_src = r'../data/pH_0-5cm_1km.tif'
lu_ref = r'../data/landuse_2023.tif'
ph_out = r'../data/pH_aligned.tif'

resample_raster(ph_src, lu_ref, ph_out)

In [ ]:
from preprocessing import load_raster

lu_data, _, _ = load_raster(lu_ref)
ph_data, _, _ = load_raster(ph_out)

print("土地利用形状:", lu_data.shape)
print("pH_aligned形状:", ph_data.shape)
print("形状是否一致？", lu_data.shape == ph_data.shape)

生成面积交叉表

In [ ]:
import sys
sys.path.append(r'../src')
from preprocessing import load_raster, classify_ph, calc_area_crosstab
import numpy as np

# 1. 读取对齐后的数据
lu_path = r'../data/landuse_2023.tif'
ph_path = r'../data/pH_aligned.tif'

lu_data, lu_transform, lu_pixel_area = load_raster(lu_path)
ph_data, _, _ = load_raster(ph_path)
ph_data = ph_data/100

# 2. pH分级
ph_class = classify_ph(ph_data)

# 3. 土地利用一级大类映射
landuse_names = {
    1: "cropland",
    2: "forest",
    3: "grass",
    4: "water",
    5: "urban",
    6: "unused land"
}

# 4. 计算面积交叉表
area_table = calc_area_crosstab(lu_data, ph_class, landuse_names, lu_pixel_area)

print("\n========== 土地利用 × pH等级 面积表（km²）==========\n")
print(area_table)

# 5. 保存为CSV
import os
os.makedirs('../outputs', exist_ok=True)
area_table.to_csv(r'../outputs/area_crosstab.csv', encoding='utf-8-sig')
print("\n面积表已保存至 outputs/area_crosstab.csv")

In [ ]:
import numpy as np
import sys
sys.path.append(r"../src")
from preprocessing import load_raster
from risk_model import calc_acid_risk
lu_path = r"../data/landuse_2023.tif"
lu_data,_,_ = load_raster(lu_path)
mask = calc_acid_risk(lu_data,None,None)
print("掩膜形状：",mask.shape)
print("耕地像元数量",np.sum(mask))

In [ ]:

import sys
sys.path.append(r'../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from preprocessing import load_raster
from risk_model import calc_acid_risk
import os

# 1. 加载数据
lu_path = r'../data/landuse_2023.tif'
ph_path = r'../data/pH_aligned.tif'
crop_path = r'../data/crop_ph.csv'

lu_data, lu_transform, _ = load_raster(lu_path)
ph_data, _, _ = load_raster(ph_path)
crop_df = pd.read_csv(crop_path)

# 计算风险
risk_array, stats = calc_acid_risk(lu_data, ph_data, crop_df, paddy_crop="水稻",dryland_crop="玉米")

print("\n酸化风险统计(双作物)：")
for k, v in stats.items():
    print(f"{k}: {v}")

#  画热力图（只显示有风险的地方）
plt.figure(figsize=(12, 10))
risk_show = np.where(~np.isnan(risk_array), risk_array, np.nan)
plt.imshow(risk_show, cmap='YlOrRd',interpolation='nearest')
plt.colorbar(label='pH deviation risk',shrink=0.8)
plt.title('Cropland Acidification Risk (paddy_crop:rice,dryland_crop:maize )',fontsize=14)
plt.axis('off')
plt.tight_layout()

os.makedirs('../outputs', exist_ok=True)
plt.savefig(r"../outputs/risk_map_final.png",dpi=150)
plt.show()

从这张图中我们就可以看出华北平原和长三角地区北部是最严重的地方

In [ ]:
import geopandas as gpd
import os
shp_dir = r"../data/county_shp"
shp_files = [f for f in os.listdir(shp_dir) if f.endswith('.shp')]
print("找到的shp文件：",shp_files)
if shp_files:
    shp_path = os.path.join(shp_dir,shp_files[0])
    gdf = gpd.read_file(shp_path)
    print("\n字段明列表：",gdf.columns.tolist())
    print("\n前两行数据：")
    print(gdf.head(2))

In [ ]:
import sys
sys.path.append(r'../src')
import pandas as pd
import matplotlib.pyplot as plt
from preprocessing import load_raster
from risk_model import calc_acid_risk, calc_county_risk

# 加载数据
lu_path = r'../data/landuse_2023.tif'
ph_path = r'../data/pH_aligned.tif'
crop_path = r'../data/crop_ph.csv'

lu_data, lu_transform, _ = load_raster(lu_path)
ph_data, _, _ = load_raster(ph_path)
crop_df = pd.read_csv(crop_path)

# 计算风险
risk_array, stats = calc_acid_risk(lu_data, ph_data, crop_df)

# 市级统计
shp_path = r'../data/city_shp/shi2022.shp'
city_risk = calc_county_risk(risk_array, lu_transform, shp_path)

print("\n========== 酸化风险最高10个市 ==========")
top10 = city_risk.sort_values("mean_risk", ascending=False).head(10)
for _, row in top10.iterrows():
    print(f"{row['county_name']}: 平均风险={row['mean_risk']}, 高风险占比={row['high_risk_ratio(%)']}%")

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)
city_risk.to_csv(
    r'../outputs/city_risk_ranking.csv', index=False, encoding="utf-8-sig"
)
print("市级风险排行已保存到 outputs/city_risk_ranking.csv")

## 分析结论

### 1. 全国耕地酸化风险概况
- 基于假设“水田种水稻、旱地种玉米”，全国耕地中约有 **54.11%** 的面积土壤 pH 偏离了作物的最适范围。
- 平均风险偏离度为 **0.30 pH 单位**，说明大部分耕地处于轻微至中度偏离状态。
- 最大偏离度达 **2.72 pH 单位**，局部地区土壤酸碱度与作物需求严重不匹配。

### 2. 高风险区域分布
- 从市级排名来看，高风险区域主要集中在 **北方地区**，尤其是宁夏、河南、河北、青海等省份。
- 风险最高的 5 个市为：石嘴山市、银川市、开封市、商丘市、沧州市。
- 这些城市的高风险面积占比均超过 **99%**，耕地几乎整体处于酸碱不适宜状态。
- 北方土壤多为碱性，而水稻和玉米的最适 pH 范围偏中性（5.5-7.5），这解释了北方城市风险值显著偏高的原因。

### 3. 政策建议
- **高风险区**：建议优先进行土壤改良，如施用酸性肥料、石膏等，降低土壤 pH 值。
- **作物调整**：在碱化严重的区域，可考虑推广更耐碱的作物品种，如甜菜、向日葵、棉花等。
- **精细化分析**：后续可结合县级数据、实际种植结构，进一步细化风险评估，为精准农业提供支持。

### 4. 局限性说明
- 作物假设基于理想情况（水田=水稻，旱地=玉米），实际种植结构可能因地区而异。
- pH 数据来自全国尺度栅格产品，可能存在局部精度不足的问题。
- 风险模型仅考虑 pH 偏离度，未纳入土壤养分、气候等其他因素。